In [1]:
from densematcher.model import MeshFeaturizer
from densematcher.utils import load_pytorch3d_mesh, recenter, get_groups_dmtx, get_uniform_SO3_RT, double_plot, get_colors, generate_colors, load_pytorch3d_mesh
from densematcher.pyFM.mesh.geometry import geodesic_distmat_dijkstra, heat_geodmat_robust
from densematcher import diffusion_net
from densematcher.diffusion_net.utils import random_rotate_points, random_rotation_matrix
import os
import numpy as np
import torch
from featup.util import pca
os.environ["INFERENCE"] = "1" # speeds up the model loading time by directly loading stuff onto GPU

/cluster/home/beellis/miniconda3/envs/densematcher_3.10/lib/python3.10/site-packages/diffusers/models/unet_2d_condition.py:20: FutureWarning: `UNet2DConditionOutput` is deprecated and will be removed in version 0.29. Importing `UNet2DConditionOutput` from `diffusers.models.unet_2d_condition` is deprecated and this will be removed in a future version. Please use `from diffusers.models.unets.unet_2d_condition import UNet2DConditionOutput`, instead.
  deprecate("UNet2DConditionOutput", "0.29", deprecation_message)
/cluster/home/beellis/miniconda3/envs/densematcher_3.10/lib/python3.10/site-packages/diffusers/models/unet_2d_condition.py:25: FutureWarning: `UNet2DConditionModel` is deprecated and will be removed in version 0.29. Importing `UNet2DConditionModel` from `diffusers.models.unet_2d_condition` is deprecated and this will be removed in a future version. Please use `from diffusers.models.unets.unet_2d_condition import UNet2DConditionModel`, instead.
  deprecate("UNet2DConditionModel",

In [2]:
width = 512 # number of channels in DiffusionNet
num_blocks = 8 # number of blocks in DiffusionNet
imsize = 384 # slightly affects accuracy, but not much

model = MeshFeaturizer(f"checkpoints/featup_imsize={imsize}_channelnorm=False_unitnorm=False_rotinv=True/final.ckpt",
                        (3, 1),
                        num_blocks,
                        width,
                        aggre_net_weights_folder="checkpoints/SDDINO_weights",
                        )


imsize 384 Using channelnorm: False Using unitnorm: False Rot inv: True


/cluster/home/beellis/miniconda3/envs/densematcher_3.10/lib/python3.10/site-packages/pytorch_lightning/utilities/distributed.py:258: LightningDeprecationWarning: `pytorch_lightning.utilities.distributed.rank_zero_only` has been deprecated in v1.8.1 and will be removed in v2.0.0. You can import it from `pytorch_lightning.utilities` instead.
  rank_zero_deprecation(


0.15627789497375488 downloaded config
LatentDiffusion: Running in eps-prediction mode
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 320, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is None and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 640, context_dim is 768 and using 8 heads.
Setting up MemoryEfficientCrossAttention. Query dim is 1280, context_dim is None and using 8 heads.
Setting up MemoryEfficient

backbone.feature_extractor.,category_head.clip.clip.,clip_head.clip.clip.
backbone.feature_projections.2.0.conv1.weight
backbone.feature_projections.2.0.shortcut.weight
backbone.feature_projections.3.0.conv1.weight
backbone.feature_projections.3.0.shortcut.weight
backbone.feature_projections.4.0.conv1.weight
backbone.feature_projections.4.0.shortcut.weight
backbone.feature_projections.5.0.conv1.weight
backbone.feature_projections.5.0.shortcut.weight
Using cache found in /cluster/home/beellis/.cache/torch/hub/facebookresearch_dinov2_main


loaded sd model


/cluster/home/beellis/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:43: UserWarning: xFormers is available (SwiGLU)
  warnings.warn("xFormers is available (SwiGLU)")
/cluster/home/beellis/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:27: UserWarning: xFormers is available (Attention)
  warnings.warn("xFormers is available (Attention)")
/cluster/home/beellis/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:33: UserWarning: xFormers is available (Block)
  warnings.warn("xFormers is available (Block)")


loaded vit model
AggregationNetwork has 5150214 parameters.
Featup has 615524 parameters.


In [3]:
# load weights into model
ckpt_file = f"checkpoints/exp_mvmatcher_imsize={imsize}_width={width}_nviews=3x1_wrecon=10.0_cutprob=0.5_blocks={num_blocks}_release_jitter=0.0/final.ckpt"
ckpt = torch.load(ckpt_file)
state_dict = {}
for key in ckpt["state_dict"].keys():
    if key.startswith("model.extractor_3d"):
        state_dict[key.removeprefix("model.extractor_3d.")] = ckpt["state_dict"][key]
model.extractor_3d.load_state_dict(state_dict)

# move model to gpu
model.to("cuda:0").half()
model.extractor_2d.featurizer.mem_eff = True # tradeoff speed to save memory by forwarding one view at a time into 2D backbone

In [4]:
def get_mesh(instance, num_views=(1, 3), random_rotation=True):
    '''
    args:
        instance: path to object folder
        num_views: number of azimuth and elevations for the view grid. Does not count the north/south poles. Total #views is num_views[0] * num_views[1] + 2
        random_rotation: if True, randomly rotate the object
    return:
        mesh_color: PyTorch3D Mesh with texture/color. Assets are normalized to 0.3 on the largest axis
        mesh_simp: PyTorch3D Mesh with remeshed geometry
        groups: list of list of int, each sublist is a semantic group (empty if groups.txt missing)
        groups_dmtx: [num_groups, num_groups] semantic distance matrix between semantic groups, $D_{semantic}$ refered in the paper (empty if no groups)
        operators: tuple of diffusionnet operators
        cameras: tuple of rotation matrices and translations for camera extrinsics
        geodesic_distance: [V, V] geodesic distance between vertices
    '''
    mesh_color = load_pytorch3d_mesh(f"{instance}/color_mesh.obj")
    mesh_simp = load_pytorch3d_mesh(f"{instance}/simple_mesh.obj")

    def _remove_unreferenced_vertices(mesh):
        faces = mesh.faces_packed()
        verts = mesh.verts_packed()
        used = torch.unique(faces)
        if used.numel() == verts.shape[0]:
            return mesh, None
        used, _ = torch.sort(used)
        idx_map = torch.full((verts.shape[0],), -1, dtype=torch.long, device=verts.device)
        idx_map[used] = torch.arange(used.shape[0], device=verts.device)
        new_verts = verts[used]
        new_faces = idx_map[faces]

        new_textures = None
        textures = mesh.textures
        if textures is not None:
            try:
                from pytorch3d.renderer.mesh.textures import TexturesVertex
                if hasattr(textures, "verts_features_list"):
                    feats = textures.verts_features_list()[0]
                elif hasattr(textures, "verts_features_packed"):
                    feats = textures.verts_features_packed()
                else:
                    feats = None
                if feats is not None and feats.shape[0] == verts.shape[0]:
                    new_textures = TexturesVertex(verts_features=[feats[used]])
            except Exception:
                new_textures = None

        from pytorch3d.structures import Meshes
        return Meshes(verts=[new_verts], faces=[new_faces], textures=new_textures), idx_map
    
    # semantic groups are optional; they're not used for inference
    groups = []
    groups_path = os.path.join(instance, "groups.txt")
    if os.path.exists(groups_path):
        with open(groups_path) as f:
            for line in f:
                line = line.strip()
                if line:
                    groups.append(list(map(int, line.split())))
    elif os.environ.get("VERBOSE", False):
        print(f"Warning: {groups_path} not found; continuing without semantic groups.")
    # remove unreferenced vertices so geodesic distance computation does not fail
    mesh_simp, remap = _remove_unreferenced_vertices(mesh_simp)
    if remap is not None:
        if groups:
            remapped_groups = []
            for group in groups:
                new_group = []
                for g in group:
                    new_g = remap[g].item()
                    if new_g >= 0:
                        new_group.append(new_g)
                if new_group:
                    remapped_groups.append(new_group)
            groups = remapped_groups
        if os.environ.get("VERBOSE", False):
            removed = int((remap < 0).sum().item())
            print(f"Warning: removed {removed} unreferenced vertices from simple_mesh.")
    if mesh_simp.textures is None:
        from pytorch3d.renderer.mesh.textures import TexturesVertex
        verts = mesh_simp.verts_packed()
        fallback_rgb = torch.full((verts.shape[0], 3), 0.7, device=verts.device, dtype=verts.dtype)
        mesh_simp.textures = TexturesVertex(verts_features=[fallback_rgb])
    geodesic_distance = heat_geodmat_robust(mesh_simp.verts_packed().numpy(), mesh_simp.faces_packed().numpy()) # [V, V] geodesic distance between vertices
    groups_dmtx = get_groups_dmtx(geodesic_distance, groups) # [num_groups, num_groups] semantic distance matrix between semantic groups, $D_{semantic}$ refered in the paper

    # move both meshes bounding box center to origin
    recenter(mesh_color, mesh_simp)
    # get rendering cameras
    bb = mesh_color.get_bounding_boxes()
    cam_dist = bb.abs().max() * (np.random.rand() * 0.5 + 2.0)

    # compute diffusionnet operators
    operators = diffusion_net.geometry.get_operators(
        mesh_simp.verts_list()[0].cpu(),
        mesh_simp.faces_list()[0].cpu(),
        k_eig=128, # default
        op_cache_dir=os.environ.get("OP_CACHE_DIR", None), # frames aren't rotation invariant but they aren't needed
        normals=mesh_simp.verts_normals_list()[0],
    )
    frames, mass, L, evals, evecs, gradX, gradY = operators # convert to dense, since dataloader workers cannot pickle sparse tensors
    operators = (frames, mass, L.to_dense(), evals, evecs, gradX.to_dense(), gradY.to_dense())
    
    # do random rotation
    if random_rotation:
        R_inv = torch.from_numpy(random_rotation_matrix()).type_as(mesh_simp.verts_packed())
    else:
        R_inv = torch.eye(3).to(frames)
    new_verts_color = torch.matmul(mesh_color.verts_padded(), R_inv)
    new_verts_simp = torch.matmul(mesh_simp.verts_padded(), R_inv)
    mesh_color = mesh_color.update_padded(new_verts_color)
    mesh_simp = mesh_simp.update_padded(new_verts_simp)
        
    # uniformly sample cameras around sphere
    Rs, ts, _, _ = get_uniform_SO3_RT(num_azimuth=num_views[0], num_elevation=num_views[1], distance=cam_dist, center=bb.mean(2))
    cameras = [Rs, ts]
    return mesh_color, mesh_simp, groups, groups_dmtx, operators, cameras, geodesic_distance


In [13]:
source_dirty_mesh, source_clean_mesh, groups1, groups_dmtx1, operators1, cameras1, geodesic_distance1 = get_mesh("DenseCorr3D/apple/63e2323f49db4793bd087e67b20ac350", random_rotation=False)
target_dirty_mesh, target_clean_mesh, groups2, groups_dmtx2, operators2, cameras2, geodesic_distance2 = get_mesh("DenseCorr3D/apple/4c19ae47dbe8468285ee53ff487fe51a", random_rotation=False)

# source_dirty_mesh, source_clean_mesh, groups1, groups_dmtx1, operators1, cameras1, geodesic_distance1 = get_mesh("HOT3D/assets/converted_new/106434519822892", random_rotation=False)
# target_dirty_mesh, target_clean_mesh, groups2, groups_dmtx2, operators2, cameras2, geodesic_distance2 = get_mesh("HOT3D/assets/converted_new/261746112525368", random_rotation=False)

/cluster/home/beellis/miniconda3/envs/densematcher_3.10/lib/python3.10/site-packages/pytorch3d/io/obj_io.py:546: UserWarning: No mtl file provided
  warnings.warn("No mtl file provided")


In [14]:
with torch.autocast("cuda"):
    with torch.no_grad():
        # you can see multiview renders and 2D features in the below directories
        os.environ["RENDER_DIR"] = "source_renders"
        f_source, _, fmv_source = model(source_dirty_mesh, source_clean_mesh, operators1, cameras1, return_mvfeatures=True)
        os.environ["RENDER_DIR"] = "target_renders"
        f_target, _, fmv_target = model(target_dirty_mesh, target_clean_mesh, operators2, cameras2, return_mvfeatures=True)

/cluster/home/beellis/miniconda3/envs/densematcher_3.10/lib/python3.10/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


In [15]:
# Normalize multiview features. 
fmv_source_normalized = fmv_source / fmv_source.norm(dim=1, keepdim=True).clamp(min=1e-5)
fmv_target_normalized = fmv_target / fmv_target.norm(dim=1, keepdim=True).clamp(min=1e-5)
# PCA 
[source_mv_pca], fit_pca_mv = pca([fmv_source_normalized[..., None, None]], use_torch_pca=False)
[target_mv_pca], _ = pca([fmv_target_normalized[..., None, None]], fit_pca=fit_pca_mv, use_torch_pca=False)
[source_pca], fit_pca = pca([f_source[..., None, None]], use_torch_pca=False)
[target_pca], _ = pca([f_target[..., None, None]], fit_pca=fit_pca, use_torch_pca=False)
source_mv_pca, target_mv_pca = source_mv_pca[..., 0, 0], target_mv_pca[..., 0, 0]
source_pca, target_pca = source_pca[..., 0, 0], target_pca[..., 0, 0]

amount of variance explained [0.67265833 0.10549282 0.05024976]
amount of variance explained [0.8338567  0.11855348 0.01828382]


## normalized $f_\text{multiview}$

In [16]:
double_plot(source_clean_mesh, target_clean_mesh, source_mv_pca.cpu().numpy(), target_mv_pca.cpu().numpy())

## normalized $f_\text{output}$

In [17]:
double_plot(source_clean_mesh, target_clean_mesh, source_pca.cpu().numpy(), target_pca.cpu().numpy())

In [18]:
from densematcher.functional_map import compute_surface_map
# tune these numbers based on the specific mesh category/how big the deformation is.
# In general a lower n_ev produces better results.
# See details of these in pyFM
n_ev = 15
maxiter = 5000
fit_params = {
            'w_descr': 1e4,
            'w_lap': 1e3, # isometric(length perservation). Set to 100-1000
            'w_dcomm': 0e0, # commutivity with discriptors
            'w_orient': 0, # Can change this to 0 for faster computation. Preserves mapping chiralty
            'w_area': 0, # area preservation
            'w_conformal': 0e1, # conformality
            'w_p2p': 0,
            'w_stochastic': 0,
            'w_ent': 1e-1,
            'w_range01': 0,
            'w_sumto1': 1e1,
            'optinit': 'zeros',
            'maxiter': maxiter,
            }
surface_map, surface_map_inv, hungarian, hungarian_precise, surface_map_icp, surface_map_inv_icp, hungarian_icp, fmap, mesh1, mesh2, surface_map_adjoint, surface_map_inv_adjoint, surface_map_icp_adjoint, surface_map_inv_icp_adjoint = \
    compute_surface_map(source_clean_mesh, target_clean_mesh, f_source.clone().detach().cpu().numpy(), f_target.clone().detach().cpu().numpy(), n_ev=n_ev, \
                        descr_type="neural", compute_extra=True, optimizer="L-BFGS-B", maxiter=maxiter, optimize_p2p=False, fit_params=fit_params)



Computing Laplacian spectrum
Computing 15 eigenvectors
	Done in 0.05 s
Computing 15 eigenvectors
	Done in 0.03 s

Computing descriptors

	512 out of 512 possible descriptors kept
	Scaling LBO commutativity weight by 2.1e-02

Optimization :
	15 Ev on source - 15 Ev on Target
	Using 512 Descriptors
	Hyperparameters :
		Descriptors preservation :1.0e+04
		Descriptors commutativity :0.0e+00
		Laplacian commutativity :1.0e+03
		Orientation preservation :0.0e+00

	Task funcall : 209, nit : 184, warnflag : CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
	Done in 1.23 seconds


In [19]:
cmap_source = get_colors(source_clean_mesh.verts_list()[0].cpu().numpy()); 
# ignore this, for computing shape only
cmap_target = cmap_source[surface_map]
cmap_target = np.zeros_like(cmap_target)
cmap_source_matched = np.zeros_like(cmap_source)

print("number of vertices", cmap_source.shape[0], cmap_target.shape[0])

# I provide 10 methods to visualize the correspondence. You can also write extra ones based on 
method = 5
# 1st way
if method == 0: # vanilla forward map
    cmap_target = cmap_source[surface_map]
elif method == 1: # vanilla forward map + icp refinment 
    cmap_target = cmap_source[surface_map_icp]
elif method == 2: # vanilla inverse map
    cmap_target[surface_map_inv] = cmap_source 
elif method == 3: # inverse map + icp refinment
    cmap_target[surface_map_inv_icp] = cmap_source 
elif method == 4: # hungarian matching based on icp refined map
    cmap_source_matched = np.zeros_like(cmap_source)
    for t, s in zip(hungarian_icp[0], hungarian_icp[1]):
        cmap_target[t] = cmap_source[s]
        cmap_source_matched[s] = cmap_source[s]
elif method == 5: # hungarian matching based on vanilla map
    cmap_target = np.zeros_like(cmap_target)
    cmap_source_matched = np.zeros_like(cmap_source)
    for t, s in zip(hungarian[0], hungarian[1]):
        cmap_target[t] = cmap_source[s]
        cmap_source_matched[s] = cmap_source[s]
elif method == 6: # adjoint forward map
    cmap_target = cmap_source[surface_map_adjoint]
elif method == 7: # adjoint inverse map
    cmap_target = cmap_source[surface_map_icp_adjoint]
elif method == 8:
    cmap_target[surface_map_inv_adjoint] = cmap_source
elif method == 9:
    cmap_target[surface_map_inv_icp_adjoint] = cmap_source

# If some vertices are not matched, fill their colors with nearest neighbor
geodist = torch.cdist(target_clean_mesh.verts_list()[0], target_clean_mesh.verts_list()[0]).numpy()
filled_mask = (cmap_target != np.zeros(3)).all(axis=1)
for vidx, value in enumerate(cmap_target):
    if ~filled_mask[vidx]:
        cmap_target[vidx] = cmap_target[filled_mask][np.argmin(geodist[vidx][filled_mask])]
    
# double_plot(target_clean_mesh,target_clean_mesh, xi_eta_faces, delta_v_norm) # for method == 7
double_plot(source_clean_mesh, target_clean_mesh, cmap_source, cmap_target)



number of vertices 1950 1204
